In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("T4 GPU")

GPU: Tesla T4 (15.6 GB)


In [ ]:
# Install all dependencies


%%capture
!pip install -q \
    transformers>=4.40.0 \
    accelerate>=0.28.0 \
    bitsandbytes>=0.43.0 \
    sentence-transformers>=2.6.0 \
    faiss-cpu>=1.8.0 \
    PyPDF2>=3.0.0 \
    gradio>=4.20.0 \
    langdetect>=1.0.9

print("All dependencies installed!")

In [ ]:
# PDF Text Extraction
from PyPDF2 import PdfReader
import re

def extract_text_from_pdf(pdf_file) -> str:
    """Extract and clean text from a PDF."""
    reader = PdfReader(pdf_file)
    pages = []

    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        if text:
            # Clean up common PDF artifacts
            text = re.sub(r'\s+', ' ', text)           # normalize whitespace
            text = re.sub(r'(?<=[a-zäöü])- (?=[a-zäöü])', '', text)  # fix hyphenation
            pages.append(text.strip())

    full_text = '\n\n'.join(pages)
    print(f"📄 Extracted {len(pages)} pages, {len(full_text):,} characters")
    return full_text

print("PDF extractor ready")

PDF extractor ready


**Basic (Naive) Chunking**

In [ ]:
# Basic Chunker (we'll upgrade this in Step 3b!)
from typing import List
import re

def basic_chunk(text: str, chunk_size: int = 400, overlap: int = 50) -> List[str]:
    """Naive chunking: split every N words with overlap.

    """
    words = text.split()
    chunks = []
    start = 0

    while start < len(words):
        end = start + chunk_size
        chunk = ' '.join(words[start:end])
        if chunk.strip():
            chunks.append(chunk)
        start = end - overlap

    print(f"[Basic] {len(chunks)} chunks "
          f"(size={chunk_size}, overlap={overlap})")
    return chunks

# testing
sample = (
    "Machine learning is a subset of artificial intelligence. "
    "It allows computers to learn from data without being explicitly programmed. "
    "The weather in Berlin is cold today. "  # topic shift!
    "Many tourists visit the Brandenburg Gate. "
    "Neural networks are inspired by biological neurons. "
    "Backpropagation is used to train deep networks. "
)
test_chunks = basic_chunk(sample, chunk_size=15, overlap=3)
for i, c in enumerate(test_chunks):
    print(f"  Chunk {i}: {c[:80]}...")
print("\n Notice: chunks mix ML and weather topics randomly!")

[Basic] 4 chunks (size=15, overlap=3)
  Chunk 0: Machine learning is a subset of artificial intelligence. It allows computers to ...
  Chunk 1: learn from data without being explicitly programmed. The weather in Berlin is co...
  Chunk 2: cold today. Many tourists visit the Brandenburg Gate. Neural networks are inspir...
  Chunk 3: by biological neurons. Backpropagation is used to train deep networks....

 Notice: chunks mix ML and weather topics randomly!


**Semantic Chunking**

In [ ]:
# Semantic Chunker - split at topic boundaries
# This uses the embedding model (loaded in Step 4, but we define the logic now)
import numpy as np

def split_into_sentences(text: str) -> List[str]:
    """Split text into sentences, handling German + English.

    Handles abbreviations (z.B., Dr., etc.) and decimal numbers.
    """
    # Protect abbreviations from being treated as sentence endings
    protected = text
    abbrevs = ['z.B.', 'bzw.', 'etc.', 'ca.', 'Dr.', 'Mr.', 'Mrs.',
               'Prof.', 'Nr.', 'd.h.', 'u.a.', 'ggf.', 'vs.']
    placeholders = {}
    for i, a in enumerate(abbrevs):
        ph = f"ABBR{i:03d}"
        placeholders[ph] = a
        protected = protected.replace(a, ph)

    # Protect decimals: 3.14 → 3DECPT14
    protected = re.sub(r'(\d)\.(\d)', r'\1DECPT\2', protected)

    # Split on .!? followed by space + uppercase letter
    raw = re.split(r'(?<=[.!?])\s+(?=[A-ZÄÖÜ])', protected)

    # Restore everything
    sentences = []
    for s in raw:
        for ph, a in placeholders.items():
            s = s.replace(ph, a)
        s = s.replace('DECPT', '.').strip()
        if s:
            sentences.append(s)
    return sentences


class SemanticChunker:
    """Split text at topic boundaries using embedding similarity.

    """

    def __init__(self, embed_model, breakpoint_percentile=25,
                 min_sentences=3, max_sentences=15):
        self.model = embed_model
        self.percentile = breakpoint_percentile
        self.min_sent = min_sentences
        self.max_sent = max_sentences

    def _find_topic_shifts(self, embeddings):
        """Compute similarity between consecutive sentences,
        find where it drops = topic shift."""
        # Cosine similarity between sentence[i] and sentence[i+1]
        similarities = np.array([
            np.dot(embeddings[i], embeddings[i+1])
            for i in range(len(embeddings) - 1)
        ])

        # Breakpoints = where similarity is unusually LOW
        threshold = np.percentile(similarities, self.percentile)
        breakpoints = [i+1 for i, s in enumerate(similarities) if s < threshold]

        return similarities, breakpoints, threshold

    def _balance_groups(self, groups):
        """Merge tiny groups, split huge ones."""
        # Merge small groups with previous
        merged = [groups[0]]
        for g in groups[1:]:
            if len(merged[-1]) < self.min_sent:
                merged[-1].extend(g)
            else:
                merged.append(g)
        if len(merged) > 1 and len(merged[-1]) < self.min_sent:
            merged[-2].extend(merged.pop())

        # Split oversized groups
        result = []
        for g in merged:
            if len(g) <= self.max_sent:
                result.append(g)
            else:
                n = (len(g) + self.max_sent - 1) // self.max_sent
                size = len(g) // n
                for i in range(0, len(g), size):
                    if g[i:i+size]:
                        result.append(g[i:i+size])
        return result

    def chunk(self, text):
        """Main method: text → semantically coherent chunks."""
        sentences = split_into_sentences(text)
        print(f" {len(sentences)} sentences found")

        if len(sentences) <= self.min_sent:
            return [' '.join(sentences)], {'similarities': [], 'breakpoints': []}

        # Embed all sentences
        print(" Embedding sentences...")
        embeddings = self.model.encode(
            sentences, show_progress_bar=True,
            batch_size=64, normalize_embeddings=True
        )

        # Find topic shifts
        similarities, breakpoints, threshold = self._find_topic_shifts(embeddings)
        print(f" Similarity: mean={similarities.mean():.3f}, "
              f"threshold={threshold:.3f}")
        print(f" Found {len(breakpoints)} topic shifts")

        # Split into groups at breakpoints
        groups, prev = [], 0
        for bp in breakpoints:
            if bp > prev:
                groups.append(sentences[prev:bp])
            prev = bp
        if prev < len(sentences):
            groups.append(sentences[prev:])

        # Balance sizes
        groups = self._balance_groups(groups)

        # Convert to text chunks
        chunks = [' '.join(g) for g in groups]
        print(f" {len(chunks)} semantic chunks created")

        debug = {
            'similarities': similarities.tolist(),
            'breakpoints': breakpoints,
            'threshold': threshold,
            'sentences': sentences,
        }
        return chunks, debug

# We'll use this after loading the embedding model in Step 4!
print(" SemanticChunker defined — will initialize after loading embeddings")

 SemanticChunker defined — will initialize after loading embeddings


**Embedding Model**

In [ ]:
# Load Multilingual Embedding Model
import numpy as np
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
dim = embed_model.get_sentence_embedding_dimension()
print(f" Model loaded! Embedding dimension: {dim}")

# Demo: See how embeddings capture meaning across languages
sentences = [
    "What is machine learning?",
    "Was ist maschinelles Lernen?",      # Same meaning, German
    "The weather is nice today.",         # Different meaning
]

embeddings = embed_model.encode(sentences, normalize_embeddings=True)

# Compute cosine similarity (dot product of normalized vectors)
for i in range(len(sentences)):
    for j in range(i+1, len(sentences)):
        sim = np.dot(embeddings[i], embeddings[j])
        print(f"  '{sentences[i][:35]}...' ↔ '{sentences[j][:35]}...'")
        print(f"  Similarity: {sim:.3f}\n")

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Model loaded! Embedding dimension: 384
  'What is machine learning?...' ↔ 'Was ist maschinelles Lernen?...'
  Similarity: 0.942

  'What is machine learning?...' ↔ 'The weather is nice today....'
  Similarity: -0.070

  'Was ist maschinelles Lernen?...' ↔ 'The weather is nice today....'
  Similarity: -0.065



**FAISS Vector Search**

In [ ]:
# FAISS Vector Store
import faiss

class VectorStore:
    """FAISS-based vector store for document chunks."""

    def __init__(self, embedding_model):
        self.model = embedding_model
        self.index = None
        self.chunks = []
        self.dimension = embedding_model.get_sentence_embedding_dimension()

    def add_documents(self, chunks: List[str]):
        """Embed and index text chunks."""
        self.chunks = chunks
        print(f" Embedding {len(chunks)} chunks...")

        embeddings = self.model.encode(
            chunks,
            show_progress_bar=True,
            batch_size=32,
            normalize_embeddings=True  # for cosine similarity
        )
        embeddings = np.array(embeddings, dtype='float32')

        # IndexFlatIP = exact inner product search
        # (= cosine similarity when vectors are normalized)
        self.index = faiss.IndexFlatIP(self.dimension)
        self.index.add(embeddings)
        print(f" FAISS index built: {self.index.ntotal} vectors")

    def search(self, query: str, top_k: int = 3) -> List[dict]:
        """Find most relevant chunks for a query."""
        if self.index is None:
            return []

        query_emb = self.model.encode(
            [query], normalize_embeddings=True
        ).astype('float32')

        scores, indices = self.index.search(query_emb, top_k)

        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < len(self.chunks):
                results.append({
                    'chunk': self.chunks[idx],
                    'score': float(score),
                    'index': int(idx)
                })
        return results

# Initialize vector store
vector_store = VectorStore(embed_model)

# NOW initialize the semantic chunker (needs the embedding model)
semantic_chunker = SemanticChunker(embed_model)
print(" Vector store + SemanticChunker ready")

 Vector store + SemanticChunker ready


**Loading LLM (Mistral-7B, 4-bit)**

In [ ]:
# Load Mistral-7B-Instruct in 4-bit
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.3"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,              # use 4-bit precision
    bnb_4bit_quant_type="nf4",      # NormalFloat4 — best for LLMs
    bnb_4bit_compute_dtype=torch.float16,  # compute in fp16
    bnb_4bit_use_double_quant=True,  # quantize the quantization constants too
)

print(f"Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model in 4-bit (first time: ~5 min download)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",          # auto-place layers on GPU/CPU
    torch_dtype=torch.float16,
)

vram_used = torch.cuda.memory_allocated() / 1e9
print(f" Model loaded! VRAM used: {vram_used:.1f} GB / 15.0 GB")

# Quick test
test_input = tokenizer("[INST] Say hello in German [/INST]", return_tensors="pt")
test_input = {k: v.to(model.device) for k, v in test_input.items()}
with torch.no_grad():
    out = model.generate(**test_input, max_new_tokens=50)
print(f"Test: {tokenizer.decode(out[0], skip_special_tokens=True)}")

Loading tokenizer...


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


 Model loaded! VRAM used: 4.6 GB / 15.0 GB
Test: Say hello in German  Hallo! (This is how you say "hello" in German.)


**RAG Pipeline**

In [ ]:
# The RAG Pipeline
from langdetect import detect

def detect_language(text: str) -> str:
    """Detect if input is German or English."""
    try:
        lang = detect(text)
        return 'de' if lang == 'de' else 'en'
    except:
        return 'en'

def build_rag_prompt(question: str, context_chunks: List[dict], lang: str) -> str:
    """Construct the prompt with retrieved context."""
    context = "\n\n---\n\n".join([c['chunk'] for c in context_chunks])

    if lang == 'de':
        system = (
            "Du bist ein hilfreicher Assistent. Beantworte die Frage NUR "
            "anhand des bereitgestellten Kontexts. Wenn der Kontext die "
            "Antwort nicht enthält, sage ehrlich, dass du es im Dokument "
            "nicht finden kannst. Antworte auf Deutsch."
        )
    else:
        system = (
            "You are a helpful assistant. Answer the question ONLY based "
            "on the provided context. If the context does not contain the "
            "answer, honestly say you cannot find it in the document. "
            "Answer in English."
        )

    return f"""[INST] {system}

### Context from the document:
{context}

### Question:
{question}

### Answer: [/INST]"""

def generate_answer(question: str, vector_store, top_k: int = 3) -> dict:
    """Full RAG: retrieve → prompt → generate."""

    # 1. Detect language
    lang = detect_language(question)

    # 2. Retrieve relevant chunks
    results = vector_store.search(question, top_k=top_k)
    if not results:
        return {'answer': 'No document loaded yet.', 'sources': [], 'language': lang}

    # 3. Build prompt with context
    prompt = build_rag_prompt(question, results, lang)

    # 4. Generate answer
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3500)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.3,       # low = more focused answers
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the NEW tokens (skip the prompt)
    generated = tokenizer.decode(
        output[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    ).strip()

    return {
        'answer': generated,
        'sources': [{'chunk_index': r['index'], 'score': r['score']}
                    for r in results],
        'language': lang
    }

print(" RAG pipeline ready!")

 RAG pipeline ready!


**End-to-End Pipeline testing**

In [ ]:
# Test with a real PDF — Compare both chunking strategies!
# Upload a PDF to Colab first (folder icon → Upload)

# Option A: Your own PDF
# pdf_path = "/content/your_file.pdf"

# Option B: Download a sample
!wget -q "https://www.gesetze-im-internet.de/gg/GG.pdf" -O /content/sample.pdf 2>/dev/null || echo "Download failed - upload your own PDF"

import os
pdf_path = "/content/sample.pdf"

if os.path.exists(pdf_path):
    # Extract text
    text = extract_text_from_pdf(pdf_path)
    print(f"Preview: {text[:200]}...\n")

    # ============================
    # COMPARE: Basic vs Semantic
    # ============================
    print("=" * 55)
    print("BASIC CHUNKING")
    print("=" * 55)
    basic_chunks = basic_chunk(text, chunk_size=400, overlap=50)
    print(f"Chunk 0 preview: {basic_chunks[0][:100]}...\n")

    print("=" * 55)
    print("SEMANTIC CHUNKING")
    print("=" * 55)
    sem_chunks, debug_info = semantic_chunker.chunk(text)
    print(f"Chunk 0 preview: {sem_chunks[0][:100]}...\n")

    # Show similarity plot data
    sims = debug_info['similarities']
    bps = debug_info['breakpoints']
    print(f"Topic shifts detected at sentence indices: {bps}")
    print(f"Similarity range: {min(sims):.3f} to {max(sims):.3f}")

    # Compare stats
    basic_lens = [len(c.split()) for c in basic_chunks]
    sem_lens = [len(c.split()) for c in sem_chunks]
    print(f"\n{'Metric':<25} {'Basic':>10} {'Semantic':>10}")
    print("-" * 48)
    print(f"{'Chunks':<25} {len(basic_chunks):>10} {len(sem_chunks):>10}")
    print(f"{'Avg words/chunk':<25} {np.mean(basic_lens):>10.0f} {np.mean(sem_lens):>10.0f}")
    print(f"{'Size std dev':<25} {np.std(basic_lens):>10.1f} {np.std(sem_lens):>10.1f}")

    # USE SEMANTIC CHUNKS for the rest of the project!
    vector_store.add_documents(sem_chunks)

    # Test questions
    print("\n" + "=" * 55)
    print("TEST: English question")
    print("=" * 55)
    result = generate_answer("What is this document about?", vector_store)
    print(f"Answer: {result['answer'][:300]}")

    print("\n" + "=" * 55)
    print("TEST: German question")
    print("=" * 55)
    result_de = generate_answer("Worum geht es in diesem Dokument?", vector_store)
    print(f"Antwort: {result_de['answer'][:300]}")
else:
    print("Please upload a PDF first!")

Download failed - upload your own PDF


EmptyFileError: Cannot read an empty file

In [14]:
import os

# Make sure the file is there
pdf_path = "/content/test_document.pdf"

if os.path.exists(pdf_path):
    print(f"✅ File found! Size: {os.path.getsize(pdf_path)} bytes")
else:
    print("❌ Upload test_document.pdf first! (folder icon → Upload)")

✅ File found! Size: 7423 bytes


In [15]:
text = extract_text_from_pdf("/content/test_document.pdf")
print(f"Preview: {text[:200]}...\n")

# Basic chunking (instant)
print("=" * 50)
print("BASIC CHUNKING")
print("=" * 50)
basic_chunks = basic_chunk(text, chunk_size=300, overlap=50)

# Semantic chunking (should take < 30 sec)
print("\n" + "=" * 50)
print("SEMANTIC CHUNKING")
print("=" * 50)
sem_chunks, debug = semantic_chunker.chunk(text)

# Load into vector store
vector_store.add_documents(sem_chunks)

# Ask a question!
print("\n" + "=" * 50)
print("ASKING: What is machine learning?")
print("=" * 50)
result = generate_answer("What is machine learning?", vector_store)
print(f"\nAnswer: {result['answer']}")

📄 Extracted 3 pages, 6,560 characters
Preview: Introduction to Artificial Intelligence What is Artificial Intelligence? Artificial Intelligence (AI) is the simulation of human intelligence processes by computer systems. These processes include lea...

BASIC CHUNKING
[Basic] 4 chunks (size=300, overlap=50)

SEMANTIC CHUNKING
 51 sentences found
 Embedding sentences...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 Similarity: mean=0.437, threshold=0.315
 Found 13 topic shifts
 9 semantic chunks created
 Embedding 9 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

 FAISS index built: 9 vectors

ASKING: What is machine learning?

Answer: Machine learning is a subset of Artificial Intelligence where computer systems are able to learn and improve from experience without being explicitly programmed. This involves algorithms that enable computers to recognize patterns, make decisions, and solve problems based on data.


In [16]:
result_de = generate_answer("Welche deutschen Unternehmen nutzen KI?", vector_store)
print(f"Antwort: {result_de['answer']}")
print(f"Language: {result_de['language']}")

Antwort: BMW, Mercedes-Benz, Volkswagen, SAP und Siemens nutzen KI.
Language: de


**Gradio UI**

In [1]:
import gradio as gr

current_doc_name = ""

def process_pdf(pdf_file):
    global current_doc_name
    if pdf_file is None:
        return " Please upload a PDF file."
    try:
        text = extract_text_from_pdf(pdf_file.name)
        if len(text.strip()) < 50:
            return " Could not extract enough text."

        sentences = split_into_sentences(text)
        if len(sentences) > 300:
            print(f" Large doc ({len(sentences)} sentences) → sentence-grouped")
            chunks = []
            for i in range(0, len(sentences), 10):
                chunk = ' '.join(sentences[i:i+10])
                if chunk.strip():
                    chunks.append(chunk)
        else:
            chunks, _ = semantic_chunker.chunk(text)

        vector_store.add_documents(chunks)
        current_doc_name = pdf_file.name.split('/')[-1]
        lang = "German" if detect_language(text[:500]) == "de" else "English"

        return (
            f" **{current_doc_name}** loaded!\n\n"
            f"• Chunks: {len(chunks)}\n"
            f"• Language: {lang}\n\n"
            f"Ask me anything! (English or German)"
        )
    except Exception as e:
        return f" Error: {str(e)}"

def chat(message, history):
    if vector_store.index is None:
        return "Please upload a PDF first! "
    result = generate_answer(message, vector_store, top_k=3)
    sources = ", ".join([f"#{s['chunk_index']}({s['score']:.2f})" for s in result['sources']])
    return f"{result['answer']}\n\n---\n📎 *Sources: {sources}*"

with gr.Blocks(title="Talk to Your PDF") as demo:
    gr.Markdown("## 📄 Talk to Your PDF\n*Multilingual RAG - powered by Mistral-7B*")

    with gr.Row():
        with gr.Column(scale=1):
            pdf_input = gr.File(label="Upload PDF", file_types=[".pdf"], type="filepath")
            upload_btn = gr.Button(" Process Document", variant="primary")
            status = gr.Markdown("Upload a PDF to start →")
            gr.Markdown("### Try asking:\n"
                       "- What is this document about?\n"
                       "- Worum geht es hier?\n"
                       "- Summarize the key points.\n"
                       "- Was sind die Hauptthemen?")

        with gr.Column(scale=2):
            chatbot = gr.Chatbot(height=450, type="messages")
            msg = gr.Textbox(placeholder="Ask about your PDF... (EN or DE)", label="Your question")
            clear = gr.Button("🗑️ Clear Chat")

    def respond(message, chat_history):
        bot_reply = chat(message, chat_history)
        chat_history.append({"role": "user", "content": message})
        chat_history.append({"role": "assistant", "content": bot_reply})
        return "", chat_history

    msg.submit(respond, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: [], outputs=[chatbot])
    upload_btn.click(fn=process_pdf, inputs=[pdf_input], outputs=[status])

demo.launch(share=True)

/tmp/ipykernel_335/256298696.py:60: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=450, type="messages")


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://438c11fc253dddf388.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
